# YOLO 物件偵測訓練

In [ ]:
PRETRAIN_MODEL = "yolo11n.pt"
LABEL_DIR = "/media/jianhua/HDD 1T/mot_annotations"
DATASET_DIR = "/home/jianhua/Desktop/dataset/SeaDronesSee_MOT"
YOLO_LABEL_DIR = "/media/jianhua/HDD 1T/yolo_obj_detect"
DATA_YAML = "data.yaml"

# PRETRAIN_MODEL = "yolo11n.pt"
# LABEL_DIR = "/home/jianhua/Desktop/dataset/mot_annotations"
# DATASET_DIR = "/home/jianhua/Desktop/dataset/SeaDronesSee_MOT"
# YOLO_LABEL_DIR = "/home/jianhua/Desktop/dataset/yolo_obj_detect"
# DATA_YAML = "data.yaml"

## 轉換格式 COCO -> YOLO

In [3]:
from ultralytics.data.converter import convert_coco

convert_coco(
    labels_dir=LABEL_DIR,
    save_dir=YOLO_LABEL_DIR
)

Annotations /home/jianhua/Desktop/dataset/mot_annotations/train.json: 100% ━━━━━━━━━━━━ 27226/27226 6.9Kit/s 3.9s0.1s
Annotations /home/jianhua/Desktop/dataset/mot_annotations/val.json: 100% ━━━━━━━━━━━━ 8579/8579 6.6Kit/s 1.3s0.1s
COCO data converted successfully.
Results saved to /home/jianhua/Desktop/dataset/yolo_obj_detect2


In [4]:
import os
import shutil

src_labels = os.path.join(YOLO_LABEL_DIR, "labels")
dst_labels = os.path.join(DATASET_DIR, "labels")

# 1️⃣ 如果 DATASET_DIR 已有 labels 就刪掉
if os.path.exists(dst_labels):
    shutil.rmtree(dst_labels)

# 2️⃣ 搬移 labels 資料夾
shutil.move(src_labels, dst_labels)

# 3️⃣ 刪掉整個 YOLO_LABEL_DIR
if os.path.exists(YOLO_LABEL_DIR):
    shutil.rmtree(YOLO_LABEL_DIR)

print("labels moved and YOLO_LABEL_DIR removed.")

# ================= 新增的部分 =================

# 定義 classes.txt 的來源與目標路徑
src_classes = "classes.txt" # 當下目錄下的 classes.txt
dst_train_dir = os.path.join(DATASET_DIR, "labels", "train")
dst_val_dir = os.path.join(DATASET_DIR, "labels", "val")

# 確保目標資料夾 train 和 val 存在 (避免 shutil.copy 找不到資料夾報錯)
os.makedirs(dst_train_dir, exist_ok=True)
os.makedirs(dst_val_dir, exist_ok=True)

# 4️⃣ 複製 classes.txt 到 train 與 val 資料夾
if os.path.exists(src_classes):
    shutil.copy(src_classes, dst_train_dir)
    shutil.copy(src_classes, dst_val_dir)
    print("classes.txt successfully copied to train and val directories.")
else:
    print(f"⚠️ 警告：在當下目錄找不到 {src_classes}，略過複製步驟。")

labels moved and YOLO_LABEL_DIR removed.
classes.txt successfully copied to train and val directories.


## 開始訓練

### YOLO Library

In [ ]:
# from ultralytics import YOLO
# import os

# model = YOLO(PRETRAIN_MODEL)  # load a pretrained model (recommended for training)

# val_classes = os.path.join(DATASET_DIR, "labels/val/classes.txt")
# train_classes = os.path.join(DATASET_DIR, "labels/train/classes.txt")

# if os.path.exists(train_classes) and os.path.exists(val_classes):
#     results = model.train(data="data.yaml", epochs=10, imgsz=640, batch=32)
# else:
#     raise FileNotFoundError(f"Missing required files or directories: {val_classes} {train_classes}")

FileNotFoundError: Missing required files or directories: /home/jianhua/Desktop/dataset/SeaDronesSee_MOT/labels/val/classes.txt /home/jianhua/Desktop/dataset/SeaDronesSee_MOT/labels/train/classes.txt

### Costum YOLO Architecture

In [2]:
import yaml
import os

def load_yolo_data(yaml_path):
    # 檢查檔案是否存在
    if not os.path.exists(yaml_path):
        print(f"找不到檔案: {yaml_path}")
        return None

    with open(yaml_path, 'r', encoding='utf-8') as f:
        try:
            # 使用 safe_load 避免執行惡意代碼，這是處理 YAML 的標準做法
            data = yaml.safe_load(f)
            return data
        except yaml.YAMLError as exc:
            print(f"讀取 YAML 時發生錯誤: {exc}")
            return None

config = load_yolo_data(DATA_YAML)

if config:
    print("--- 成功讀取資料集配置 ---")
    
    # 1. 取得類別數量
    nc = config.get('nc', 0)
    print(f"類別數量 (nc): {nc}")
    
    # 2. 取得類別名稱列表
    names = config.get('names', [])
    print(f"類別名稱: {names}")
    
    # 3. 取得訓練/驗證路徑
    train_path = config.get('train', '未設定')
    val_path = config.get('val', '未設定')
    print(f"訓練集路徑: {train_path}")
    print(f"驗證集路徑: {val_path}")

    # 如果 names 是字典格式 {0: 'person', 1: 'dog'}，可以這樣處理
    if isinstance(names, dict):
        class_list = list(names.values())
        print(f"轉換後的類別列表: {class_list}")

--- 成功讀取資料集配置 ---
類別數量 (nc): 4
類別名稱: {0: 'swimmer', 1: 'swimmer_with_life_jacket', 2: 'boat', 3: 'life_jacket'}
訓練集路徑: images/train
驗證集路徑: images/val
轉換後的類別列表: ['swimmer', 'swimmer_with_life_jacket', 'boat', 'life_jacket']


In [ ]:
# from arch.yolo11 import yolo_v11_n
# from torchinfo import summary

# # yolo_model = yolo_v11_n(num_classes=nc)
# yolo_model = yolo_v11_n(num_classes=4)

# # input_size 格式為 (Batch_size, Channels, Height, Width)
# # YOLO 通常使用 640x640 的輸入
# summary(yolo_model, input_size=(1, 3, 640, 640))

/home/jianhua/miniconda3/envs/torch2/lib/python3.9/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /opt/conda/conda-bld/pytorch_1678402374358/work/aten/src/ATen/native/TensorShape.cpp:3483.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Layer (type:depth-idx)                                            Output Shape              Param #
YOLO                                                              [1, 8, 8400]              --
├─DarkNet: 1-1                                                    [1, 128, 80, 80]          --
│    └─Sequential: 2-1                                            [1, 16, 320, 320]         --
│    │    └─Conv: 3-1                                             [1, 16, 320, 320]         464
│    └─Sequential: 2-2                                            [1, 64, 160, 160]         --
│    │    └─Conv: 3-2                                             [1, 32, 160, 160]         4,672
│    │    └─CSP: 3-3                                              [1, 64, 160, 160]         6,640
│    └─Sequential: 2-3                                            [1, 128, 80, 80]          --
│    │    └─Conv: 3-4                                             [1, 64, 80, 80]           36,992
│    │    └─CSP: 3-5              

In [ ]:
"""
YOLOv11 Custom Training Script
Based on: https://github.com/jahongir7174/YOLOv11-pt
"""

import copy
import csv
import glob
import os
from types import SimpleNamespace

import torch
import tqdm
from torch.utils.data import DataLoader

from arch.yolo11 import yolo_v11_n
from utils import util
from utils.dataset import create_dataloader

# ─────────────────────────────────────────────────────────────
# 所有設定直接在這裡修改
# ─────────────────────────────────────────────────────────────

TRAIN_IMG_DIR = os.path.join(DATASET_DIR, "images/train")
VAL_IMG_DIR   = os.path.join(DATASET_DIR, "images/val")

INPUT_SIZE  = 640
BATCH_SIZE  = 16
EPOCHS      = 5
WORKERS     = 4
NUM_CLASSES = 4

PARAMS = {
    # ── 類別 ──────────────────────────────────────────────
    "names": [f"class{i}" for i in range(NUM_CLASSES)],

    # ── 優化器 ────────────────────────────────────────────
    "min_lr":        1e-3,
    "max_lr":        1e-2,
    "momentum":      0.937,
    "weight_decay":  5e-4,
    "warmup_epochs": 3,
    "lrf":           0.01,

    # ── Loss 權重 ─────────────────────────────────────────
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,

    # ── Dataset 增強參數（Dataset.__getitem__ 會用到） ────
    "mosaic":    1.0,   # mosaic 機率
    "mix_up":    0.1,   # mixup 機率
    "hsv_h":     0.015, # HSV Hue 增強幅度
    "hsv_s":     0.7,   # HSV Saturation 增強幅度
    "hsv_v":     0.4,   # HSV Value 增強幅度
    "degrees":   0.0,   # 旋轉角度範圍 (±degrees)
    "scale":     0.5,   # 縮放範圍 (1±scale)
    "shear":     0.0,   # 剪切角度範圍
    "translate": 0.1,   # 平移範圍 (比例)
    "flip_ud":   0.0,   # 上下翻轉機率
    "flip_lr":   0.5,   # 左右翻轉機率
}

WEIGHTS_DIR = "weights"

# ─────────────────────────────────────────────────────────────
# 相容新舊版 PyTorch 的 AMP 工具
# ─────────────────────────────────────────────────────────────

def get_grad_scaler():
    return torch.cuda.amp.GradScaler()

# ─────────────────────────────────────────────────────────────

@torch.no_grad()
def test(params, model=None):
    # ── DataLoader ─────────────────────────────────────────
    val_loader = create_dataloader(
        img_folder = VAL_IMG_DIR,
        input_size = INPUT_SIZE,
        batch_size = 8,
        augment    = False,   # val 不做增強
        shuffle    = False,   # val 不打亂
        hyp_params = params,
    )
 
    # ── 載入模型（若沒有從外部傳入）──────────────────────
    plot = False
    if model is None:
        plot  = True
        ckpt  = torch.load(f"{WEIGHTS_DIR}/best.pt", map_location="cuda")
        model = ckpt["model"].float().fuse()
 
    model.half()
    model.eval()
 
    # ── 評估設定 ───────────────────────────────────────────
    iou_v  = torch.linspace(start=0.5, end=0.95, steps=10).cuda()  # mAP@0.5:0.95
    n_iou  = iou_v.numel()
 
    m_pre  = 0
    m_rec  = 0
    map50  = 0
    mean_ap = 0
    metrics = []
 
    p_bar = tqdm.tqdm(val_loader,
                      desc=("%10s" * 5) % ("", "precision", "recall", "mAP50", "mAP"))
 
    for samples, targets in p_bar:
        samples = samples.cuda().half() / 255.0   # uint8 → fp16, 0~1
 
        _, _, h, w = samples.shape
        scale = torch.tensor((w, h, w, h)).cuda()
 
        # ── Inference ──────────────────────────────────────
        outputs = model(samples)
 
        # ── NMS ────────────────────────────────────────────
        outputs = util.non_max_suppression(outputs)
 
        # ── Metrics ────────────────────────────────────────
        for i, output in enumerate(outputs):
            idx = targets["idx"] == i
            cls = targets["cls"][idx].cuda()
            box = targets["box"][idx].cuda()
 
            metric = torch.zeros(output.shape[0], n_iou, dtype=torch.bool).cuda()
 
            if output.shape[0] == 0:
                if cls.shape[0]:
                    metrics.append((metric, *torch.zeros((2, 0)).cuda(), cls.squeeze(-1)))
                continue
 
            if cls.shape[0]:
                target = torch.cat(tensors=(cls, util.wh2xy(box) * scale), dim=1)
                metric = util.compute_metric(output[:, :6], target, iou_v)
 
            metrics.append((metric, output[:, 4], output[:, 5], cls.squeeze(-1)))
 
    # ── 計算最終指標 ────────────────────────────────────────
    metrics = [torch.cat(x, dim=0).cpu().numpy() for x in zip(*metrics)]
    if len(metrics) and metrics[0].any():
        tp, fp, m_pre, m_rec, map50, mean_ap = util.compute_ap(
            *metrics, plot=plot, names=params["names"]
        )
 
    print(("%10s" + "%10.3g" * 4) % ("", m_pre, m_rec, map50, mean_ap))
 
    model.float()   # 還原 fp32 供後續訓練使用
    return mean_ap, map50, m_rec, m_pre


def train(params):
    os.makedirs(WEIGHTS_DIR, exist_ok=True)

    # ── Model ──────────────────────────────────────────────
    model = yolo_v11_n(NUM_CLASSES)
    model.cuda()

    # ── Optimizer ──────────────────────────────────────────
    accumulate = max(round(64 / BATCH_SIZE), 1)
    params = params.copy()
    params["weight_decay"] *= BATCH_SIZE * accumulate / 64

    optimizer = torch.optim.SGD(
        util.set_params(model, params["weight_decay"]),
        params["min_lr"],
        params["momentum"],
        nesterov=True,
    )

    # ── EMA ────────────────────────────────────────────────
    ema = util.EMA(model)

    # ── DataLoader ─────────────────────────────────────────
    train_loader = create_dataloader(
        img_folder = TRAIN_IMG_DIR,
        input_size = INPUT_SIZE,
        batch_size = BATCH_SIZE,
        augment    = True,
        shuffle    = True,
        hyp_params = params,
    )
    num_steps = len(train_loader)

    # ── Scheduler ──────────────────────────────────────────
    args = SimpleNamespace(
        epochs     = EPOCHS,
        local_rank = 0,
        world_size = 1,
    )
    scheduler = util.LinearLR(args, params, num_steps)

    # ── Loss & AMP ─────────────────────────────────────────
    criterion = util.ComputeLoss(model, params)
    amp_scale = torch.cuda.amp.GradScaler()

    # ── Training Loop ──────────────────────────────────────
    best = 0.0

    with open(f"{WEIGHTS_DIR}/step.csv", "w") as log:
        logger = csv.DictWriter(log, fieldnames=['epoch',
                                                 'box', 'cls', 'dfl',
                                                 'Recall', 'Precision', 'mAP@50', 'mAP'])
        logger.writeheader()

        for epoch in range(EPOCHS):
            model.train()

            # 最後 10 epoch 關閉 mosaic
            if EPOCHS - epoch == 10:
                train_loader.dataset.mosaic = False

            avg_box = util.AverageMeter()
            avg_cls = util.AverageMeter()
            avg_dfl = util.AverageMeter()

            optimizer.zero_grad()

            print(("\n" + "%10s" * 5) % ("epoch", "memory", "box", "cls", "dfl"))
            p_bar = tqdm.tqdm(enumerate(train_loader), total=num_steps)

            for i, (samples, targets) in p_bar:
                step = i + num_steps * epoch
                scheduler.step(step, optimizer)

                samples = samples.cuda().float() / 255.0

                with torch.autocast("cuda"):
                    outputs = model(samples)

                loss_box, loss_cls, loss_dfl = criterion(outputs, targets)
 
                avg_box.update(loss_box.item(), samples.size(0))
                avg_cls.update(loss_cls.item(), samples.size(0))
                avg_dfl.update(loss_dfl.item(), samples.size(0))

                loss_box *= BATCH_SIZE
                loss_cls *= BATCH_SIZE
                loss_dfl *= BATCH_SIZE

                amp_scale.scale(loss_box + loss_cls + loss_dfl).backward()

                if step % accumulate == 0:
                    amp_scale.step(optimizer)
                    amp_scale.update()
                    optimizer.zero_grad()
                    ema.update(model)

                torch.cuda.synchronize()

                mem = f"{torch.cuda.memory_reserved() / 1e9:.3g}G"
                s = ("%10s" * 2 + "%10.4g" * 3) % (
                    f"{epoch + 1}/{EPOCHS}", mem,
                    avg_box.avg, avg_cls.avg, avg_dfl.avg,
                )
                p_bar.set_description(s)

            # ── 記錄 ───────────────────────────────────────
            last = test(params, ema.ema)
            logger.writerow({'epoch': str(epoch + 1).zfill(3),
                             'box': str(f'{avg_box.avg:.3f}'),
                             'cls': str(f'{avg_cls.avg:.3f}'),
                             'dfl': str(f'{avg_dfl.avg:.3f}'),
                             'mAP': str(f'{last[0]:.3f}'),
                             'mAP@50': str(f'{last[1]:.3f}'),
                             'Recall': str(f'{last[2]:.3f}'),
                             'Precision': str(f'{last[3]:.3f}')})
            log.flush()

            # ── 儲存 ───────────────────────────────────────
            save = {"epoch": epoch + 1, "model": copy.deepcopy(ema.ema)}
            torch.save(save, f"{WEIGHTS_DIR}/last.pt")

            if avg_box.avg < best or best == 0.0:
                best = avg_box.avg
                torch.save(save, f"{WEIGHTS_DIR}/best.pt")
            del save

    util.strip_optimizer(f"{WEIGHTS_DIR}/best.pt")
    util.strip_optimizer(f"{WEIGHTS_DIR}/last.pt")


if __name__ == "__main__":
    util.setup_seed()
    util.setup_multi_processes()
    train(PARAMS)
    torch.cuda.empty_cache()

TypeError: train() missing 1 required positional argument: 'params'

In [ ]:
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import DataLoader

# # ==========================================
# # 1. 替換為該 Repo 的模型、資料集與損失函數
# # ==========================================
# # 以下為示意 Import，你需要根據 main.py 或 nets/ 內的實際名稱修改
# # from nets.yolo11 import YOLOv11
# # from utils.dataloader import YOLODataset
# # from nets.loss import YOLOLoss 

# def train_one_epoch(model, dataloader, optimizer, criterion, device, epoch):
#     model.train()
#     total_loss = 0
    
#     for batch_idx, (images, targets) in enumerate(dataloader):
#         images, targets = images.to(device), targets.to(device)
        
#         # 前向傳播
#         outputs = model(images)
        
#         # 計算損失 (YOLO 的 Loss 通常包裝好了，直接丟入預測和標籤)
#         loss = criterion(outputs, targets)
        
#         # 反向傳播與參數更新
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()
        
#         total_loss += loss.item()
        
#         # 每 10 個 batch 印出一次進度
#         if batch_idx % 10 == 0:
#             print(f"Train Epoch: {epoch} [{batch_idx}/{len(dataloader)}] \t Loss: {loss.item():.4f}")
            
#     return total_loss / len(dataloader)

# def test_model(model, dataloader, criterion, device):
#     model.eval()
#     test_loss = 0
    
#     # 測試階段不需要計算梯度
#     with torch.no_grad():
#         for images, targets in dataloader:
#             images, targets = images.to(device), targets.to(device)
            
#             outputs = model(images)
#             loss = criterion(outputs, targets)
#             test_loss += loss.item()
            
#             # 這裡通常會呼叫 mAP 計算函數 (如 utils.metrics 裡的邏輯)
#             # calculate_map(outputs, targets)
            
#     avg_loss = test_loss / len(dataloader)
#     print(f"\nTest set: Average loss: {avg_loss:.4f}\n")
#     return avg_loss

# if __name__ == "__main__":
#     # 設定硬體設備
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     print(f"Using device: {device}")
    
#     # ==========================================
#     # 2. 超參數設定與實例化
#     # ==========================================
#     epochs = 50
#     batch_size = 16
#     learning_rate = 1e-3
    
#     """
#     # 註解取消後，替換為該專案實際的資料集與模型呼叫方式
    
#     # 準備 DataLoader
#     train_dataset = YOLODataset(annotation_lines_train, input_shape, ...)
#     test_dataset  = YOLODataset(annotation_lines_test, input_shape, ...)
    
#     train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
#     test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
#     # 初始化模型與損失函數
#     model = YOLOv11(num_classes=80).to(device)
#     criterion = YOLOLoss()
    
#     # 設定優化器
#     optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
#     # ==========================================
#     # 3. 執行主迴圈
#     # ==========================================
#     for epoch in range(1, epochs + 1):
#         print(f"--- Starting Epoch {epoch} ---")
#         train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, epoch)
#         test_loss = test_model(model, test_loader, criterion, device)
        
#         # 可以加上儲存權重的邏輯
#         # torch.save(model.state_dict(), f"logs/ep{epoch:03d}-loss{train_loss:.3f}.pth")
#     """